# 📉 Customer Churn Prediction
**Data Analyst Internship Project**

Predicting which telecom customers are likely to churn using EDA and a Random Forest classifier on the IBM Telco dataset (7,043 customers, 21 features).

---

**Pipeline Overview:**
1. Import Libraries
2. Load & Inspect Data
3. Data Cleaning
4. Exploratory Data Analysis (EDA)
5. Feature Encoding
6. Train-Test Split
7. Model Training — Random Forest
8. Evaluation & Metrics
9. Feature Importance Analysis
10. Business Insights & Conclusion

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, auc
)

# Display settings
pd.set_option('display.max_columns', 25)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded successfully.')

## 2. Load & Inspect Data

In [ ]:
# Update path if needed
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print('Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
print('Dataset Info:')
df.info()
print('\nMissing Values per Column:')
print(df.isnull().sum())
print('\nChurn Rate:')
print(df['Churn'].value_counts(normalize=True).mul(100).round(2).astype(str) + '%')

## 3. Data Cleaning

In [ ]:
# TotalCharges is stored as object — convert to numeric
# Blank strings become NaN (errors='coerce')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

missing = df['TotalCharges'].isnull().sum()
print(f'Missing TotalCharges after conversion: {missing}')

# Fill missing values with column median (robust to outliers)
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
print(f'Missing TotalCharges after imputation: {df["TotalCharges"].isnull().sum()}')

# Drop customerID — not a predictive feature
df = df.drop(columns=['customerID'], errors='ignore')
print('\nCleaned dataset shape:', df.shape)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# --- 4.1 Churn Distribution ---
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
ax = sns.countplot(x='Churn', data=df, palette=['#2196F3', '#F44336'], ax=axes[0])
ax.bar_label(ax.containers[0], fmt='%d')
axes[0].set_title('Customer Churn Distribution', fontsize=13)
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')

# Pie chart
churn_counts = df['Churn'].value_counts()
axes[1].pie(churn_counts, labels=['No Churn', 'Churned'],
            autopct='%1.1f%%', colors=['#2196F3', '#F44336'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Churn Proportion', fontsize=13)

plt.tight_layout()
plt.show()

In [ ]:
# --- 4.2 Contract Type vs Churn ---
plt.figure(figsize=(9, 5))
ax = sns.countplot(x='Contract', hue='Churn', data=df,
                   palette=['#2196F3', '#F44336'],
                   order=['Month-to-month', 'One year', 'Two year'])
plt.title('Contract Type vs Churn', fontsize=14)
plt.xlabel('Contract Type')
plt.ylabel('Number of Customers')
plt.legend(title='Churned', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()
print('Insight: Month-to-month contracts have the highest churn rate.')

In [ ]:
# --- 4.3 Payment Method vs Churn ---
plt.figure(figsize=(11, 5))
sns.countplot(x='PaymentMethod', hue='Churn', data=df,
              palette=['#2196F3', '#F44336'])
plt.title('Payment Method vs Churn', fontsize=14)
plt.xlabel('Payment Method')
plt.ylabel('Number of Customers')
plt.xticks(rotation=20, ha='right')
plt.legend(title='Churned', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()
print('Insight: Electronic check users churn the most.')

In [ ]:
# --- 4.4 Tenure Distribution by Churn ---
plt.figure(figsize=(9, 5))
sns.histplot(data=df, x='tenure', hue='Churn', bins=30, kde=True,
             palette=['#2196F3', '#F44336'])
plt.title('Tenure Distribution by Churn', fontsize=14)
plt.xlabel('Tenure (months)')
plt.ylabel('Count')
plt.tight_layout()
plt.show()
print('Insight: New customers (low tenure) are far more likely to churn.')

In [ ]:
# --- 4.5 Monthly Charges vs Churn ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(x='Churn', y='MonthlyCharges', data=df,
            palette=['#2196F3', '#F44336'], ax=axes[0])
axes[0].set_title('Monthly Charges vs Churn', fontsize=13)
axes[0].set_xticklabels(['No Churn', 'Churned'])

sns.violinplot(x='Churn', y='TotalCharges', data=df,
               palette=['#2196F3', '#F44336'], ax=axes[1])
axes[1].set_title('Total Charges vs Churn', fontsize=13)
axes[1].set_xticklabels(['No Churn', 'Churned'])

plt.tight_layout()
plt.show()
print('Insight: Churned customers tend to have higher monthly charges.')

In [ ]:
# --- 4.6 Correlation Heatmap ---
df_encoded = df.copy()
le = LabelEncoder()
for col in df_encoded.select_dtypes(include='object').columns:
    df_encoded[col] = le.fit_transform(df_encoded[col])

plt.figure(figsize=(15, 10))
sns.heatmap(
    df_encoded.corr(),
    cmap='coolwarm',
    annot=False,
    linewidths=0.3,
    vmin=-1, vmax=1
)
plt.title('Feature Correlation Matrix', fontsize=15)
plt.tight_layout()
plt.show()

## 5. Feature Encoding & Model Preparation

In [ ]:
# Encode all categorical columns
df_model = df.copy()
le = LabelEncoder()
for col in df_model.select_dtypes(include='object').columns:
    df_model[col] = le.fit_transform(df_model[col])

# Features and target
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']   # already 0/1 after LabelEncoder

print('Features shape:', X.shape)
print('Target distribution:')
print(y.value_counts().rename({0: 'No Churn', 1: 'Churned'}))

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # preserves churn ratio in both splits
)

print(f'Training samples : {X_train.shape[0]:,}')
print(f'Testing samples  : {X_test.shape[0]:,}')
print(f'\nChurn rate in train : {y_train.mean()*100:.1f}%')
print(f'Churn rate in test  : {y_test.mean()*100:.1f}%')

## 7. Model Training — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print('Model training complete.')

## 8. Model Evaluation

In [ ]:
# Comprehensive Metrics
metrics = {
    'Accuracy' : accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall'   : recall_score(y_test, y_pred),
    'F1 Score' : f1_score(y_test, y_pred)
}

print('=' * 38)
print('      MODEL PERFORMANCE METRICS')
print('=' * 38)
for k, v in metrics.items():
    print(f'  {k:<12}: {v:.4f}  ({v*100:.2f}%)')
print('=' * 38)

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['No Churn', 'Churned'],
    yticklabels=['No Churn', 'Churned']
)
plt.title('Confusion Matrix', fontsize=14)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()

In [ ]:
# Full Classification Report
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churned']))

In [ ]:
# ROC Curve & AUC
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Classifier')
plt.fill_between(fpr, tpr, alpha=0.08, color='darkorange')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve — Churn Prediction Model', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.tight_layout()
plt.show()

print(f'AUC Score: {roc_auc:.4f}')

## 9. Feature Importance Analysis

In [ ]:
importance = pd.Series(rf.feature_importances_, index=X.columns)
top10 = importance.sort_values(ascending=False).head(10)

print('Top 10 Features Driving Churn:')
print(top10.to_string())

plt.figure(figsize=(10, 6))
colors = ['#F44336' if i < 3 else '#FF9800' if i < 6 else '#2196F3'
          for i in range(len(top10))]
top10.sort_values().plot(kind='barh', color=colors[::-1], edgecolor='white')
plt.title('Top 10 Features Affecting Customer Churn', fontsize=14)
plt.xlabel('Feature Importance Score')
plt.tight_layout()
plt.show()

## 10. Business Insights & Conclusion

### Summary of Findings

| Finding | Detail |
|---------|--------|
| **Churn Rate** | ~26.5% of customers churned |
| **Highest Risk Segment** | Month-to-month contract customers |
| **Price Driver** | Churned customers have higher monthly charges |
| **Loyalty Signal** | Tenure >12 months → dramatically lower churn |
| **Payment Red Flag** | Electronic check users churn most |
| **Top Predictors** | TotalCharges, tenure, MonthlyCharges, Contract |

### Actionable Recommendations

1. **Retention offers for month-to-month customers** — Incentivize contract upgrades with discounts or loyalty perks.
2. **Onboarding program for new customers** — Focus on the first 12 months with proactive support.
3. **Auto-pay enrollment campaign** — Convert electronic check users to automatic payment to reduce friction and churn.
4. **Bundle value-added services** — Package `TechSupport` and `OnlineSecurity` at a lower price to improve stickiness.
5. **Price review for high-charge customers** — Targeted loyalty discounts for customers at churn risk.

### Conclusion

The **Random Forest classifier** successfully identified customers likely to churn with ~80–82% accuracy. Feature importance analysis revealed that `TotalCharges`, `tenure`, `MonthlyCharges`, and `Contract` type are the strongest predictors — all of which align with intuitive business understanding and provide clear levers for a retention strategy.

---
*Data Analyst Internship Project | Built with Python, Pandas, Scikit-learn & Seaborn*